# Empty Dock: Visualizations

Builds the four portfolio charts from the **gold** tables and exports each one as a standalone, interactive HTML page.

| # | Chart | Source table |
|---|---|---|
| 1 | City pulse: share of stations empty / full by hour | `gold.station_hour` |
| 2 | Station types map | `gold.station_profile` |
| 3 | Animated hourly map: bikes ÷ docks through the day | `gold.station_hour` + `silver.station_info` |
| 4 | Heatmap: the 40 most-often-empty stations by hour | `gold.station_hour` + `gold.station_profile` |

**Before running:** the pipeline has run with the operational-status fix, so `station_profile` contains the `'Out of service'` category. Out-of-service stations are excluded from every chart.

## 0. Setup

In [0]:
%pip install -q "plotly>=5.24"
dbutils.library.restartPython()

In [0]:
import os

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# ---- Config ---------------------------------------------------------------
CATALOG    = "citibike"          # change to "workspace" if that's your catalog
GOLD       = f"{CATALOG}.gold"
SILVER     = f"{CATALOG}.silver"
EXPORT_DIR = f"/Volumes/{CATALOG}/gold/exports"

spark.sql(f"CREATE VOLUME IF NOT EXISTS {GOLD}.exports")
os.makedirs(EXPORT_DIR, exist_ok=True)

### Shared style

One template, palette and set of helpers, so all four charts look like they belong together.

In [0]:
# ---- Palette ----------------------------------------------------------------
FONT      = "Inter, -apple-system, 'Segoe UI', Helvetica, Arial, sans-serif"
INK       = "#1f2933"
MUTED     = "#6b7280"
GRID      = "#e5e7eb"
EMPTY_RED = "#d64545"
FULL_BLUE = "#2b6cb0"

STATION_TYPE_COLORS = {
    "Residential: drains AM, fills PM": "#2b6cb0",
    "Job center: fills AM, drains PM":  "#dd6b20",
    "Chronically empty":                "#d64545",
    "Chronically full":                 "#6b46c1",
    "Balanced":                         "#a0aec0",
}

HOUR_LABELS = [f"{(h % 12) or 12}{'am' if h < 12 else 'pm'}" for h in range(24)]
NYC_CENTER  = dict(lat=40.735, lon=-73.965)
MAP_STYLE   = "carto-positron"

# ---- Plotly template --------------------------------------------------------
pio.templates["empty_dock"] = go.layout.Template(
    layout=dict(
        font=dict(family=FONT, size=13, color=INK),
        paper_bgcolor="white",
        plot_bgcolor="white",
        title=dict(x=0.02, xanchor="left", y=0.97, yanchor="top"),
        xaxis=dict(showgrid=False, linecolor=GRID, ticks="outside", tickcolor=GRID),
        yaxis=dict(gridcolor=GRID, zeroline=False, linecolor=GRID),
        legend=dict(orientation="h", yanchor="bottom", y=1.0, xanchor="left", x=0, title_text=""),
        margin=dict(l=60, r=30, t=120, b=80),
        hoverlabel=dict(font_family=FONT, bgcolor="white"),
    )
)
pio.templates.default = "simple_white+empty_dock"


# ---- Helpers ----------------------------------------------------------------
def add_titles(fig, title, subtitle):
    """Bold title with a muted subtitle underneath."""
    fig.update_layout(title_text=(
        f"<b style='font-size:21px'>{title}</b><br>"
        f"<span style='font-size:13px;color:{MUTED}'>{subtitle}</span>"
    ))
    return fig


def add_source(fig, note, y=-0.10):
    """Small source/footnote line under the chart."""
    fig.add_annotation(
        text=note, xref="paper", yref="paper", x=0, y=y,
        xanchor="left", yanchor="top", showarrow=False,
        font=dict(size=11, color=MUTED),
    )
    return fig


def export(fig, name):
    """Save as a standalone interactive HTML page (Plotly loaded from CDN keeps files small)."""
    path = f"{EXPORT_DIR}/{name}.html"
    fig.write_html(
        path,
        include_plotlyjs="cdn",
        full_html=True,
        config={"displaylogo": False, "responsive": True, "scrollZoom": True},
    )
    print(f"Saved {path}  ({os.path.getsize(path) / 1e6:.2f} MB)")

### Data coverage

Used in every chart's footnote, so readers know exactly what period the data covers.

In [0]:
cov = spark.sql(f"""
    SELECT min(hour_ny)                                  AS first_hour,
           max(hour_ny)                                  AS last_hour,
           count(DISTINCT date(hour_ny))                 AS n_days,
           count(DISTINCT station_id)                    AS n_stations
    FROM {GOLD}.station_hour
""").first()

COVERAGE    = f"{cov.first_hour:%b %d, %Y} to {cov.last_hour:%b %d, %Y} ({cov.n_days} days)"
SOURCE_NOTE = (f"Source: Citi Bike GBFS station feed, polled every 15 min · {COVERAGE} · "
               f"Out-of-service stations excluded · Hours in NYC local time")
print(SOURCE_NOTE)

## 1. City pulse

Average share of station-time spent **empty** vs **full**, by hour of day. Commute peaks are shaded.

In [0]:
def build_pulse(df, source_note):
    df = df.set_index("hour").reindex(range(24)).reset_index()   # guarantee all 24 hours, in order

    fig = go.Figure()
    for col, name, color in [("pct_empty", "Empty", EMPTY_RED), ("pct_full", "Full", FULL_BLUE)]:
        fig.add_trace(go.Scatter(
            x=df["hour"], y=df[col], name=name,
            mode="lines+markers",
            line=dict(color=color, width=3, shape="spline"),
            marker=dict(size=6),
            customdata=HOUR_LABELS,
            hovertemplate=f"<b>%{{customdata}}</b><br>{name}: %{{y:.1%}}<extra></extra>",
        ))

    for x0, x1, label in [(7, 9, "AM rush"), (16, 18, "PM rush")]:
        fig.add_vrect(
            x0=x0 - 0.5, x1=x1 + 0.5, fillcolor="#f3f4f6", opacity=0.8,
            layer="below", line_width=0,
            annotation_text=label, annotation_position="top left",
            annotation_font=dict(size=11, color=MUTED),
        )

    fig.update_xaxes(tickvals=list(range(0, 24, 3)), ticktext=HOUR_LABELS[::3],
                     range=[-0.5, 23.5], title_text="Hour of day")
    fig.update_yaxes(tickformat=".0%", rangemode="tozero", title_text="Share of station-time")
    fig.update_layout(height=540, hovermode="x unified")

    add_titles(fig, "When NYC's bike stations run dry",
               "Share of time Citi Bike stations sit empty (no bikes) or full (no free docks), by hour")
    add_source(fig, source_note, y=-0.18)
    return fig


pulse_df = spark.sql(f"""
    SELECT hour(h.hour_ny)  AS hour,
           avg(h.pct_empty) AS pct_empty,
           avg(h.pct_full)  AS pct_full
    FROM {GOLD}.station_hour h
    JOIN {GOLD}.station_profile p USING (station_id)
    WHERE p.station_type <> 'Out of service'
    GROUP BY 1
    ORDER BY 1
""").toPandas()

fig_pulse = build_pulse(pulse_df, SOURCE_NOTE)
fig_pulse.show()
export(fig_pulse, "01_city_pulse")

## 2. Station types map

Each station classified by its daily pattern (see `gold.station_profile` for the rules). Legend shows the station count per type.

In [0]:
def build_station_types(df, source_note):
    counts = df["station_type"].value_counts()
    order  = [t for t in STATION_TYPE_COLORS if t in counts.index]

    fig = px.scatter_map(
        df, lat="lat", lon="lon",
        color="station_type",
        color_discrete_map=STATION_TYPE_COLORS,
        category_orders={"station_type": order},
        hover_name="name",
        hover_data={"lat": False, "lon": False, "station_type": False,
                    "capacity": True, "am_empty": ":.0%", "pm_empty": ":.0%", "day_full": ":.0%"},
        labels={"capacity": "Docks", "am_empty": "Empty 7-10am",
                "pm_empty": "Empty 4-7pm", "day_full": "Full (all day)"},
        center=NYC_CENTER, zoom=11, map_style=MAP_STYLE, height=780,
    )
    fig.update_traces(marker=dict(size=8, opacity=0.85))
    fig.for_each_trace(lambda t: t.update(name=f"{t.name}  ({counts[t.name]:,})"))
    fig.update_layout(margin=dict(l=10, r=10, t=150, b=50), legend=dict(y=1.0, font=dict(size=12)))

    add_titles(fig, "What kind of station is it?",
               "Residential stations drain in the morning and refill at night; job centers do the opposite")
    add_source(fig, source_note, y=-0.02)
    return fig


types_df = spark.sql(f"""
    SELECT name, lat, lon, capacity, station_type, am_empty, pm_empty, day_empty, day_full
    FROM {GOLD}.station_profile
    WHERE station_type <> 'Out of service'
""").toPandas()

fig_types = build_station_types(types_df, SOURCE_NOTE)
fig_types.show()
export(fig_types, "02_station_types_map")

## 3. Animated hourly map

Every station colored by **fill ratio** (bikes ÷ docks) for each hour of the day. Press ▶ to watch the city drain and refill. The color scale is locked to 0–100% so frames are comparable.

In [0]:
def build_hourly_animation(df, source_note):
    df = df.copy()
    df["fill_ratio"] = df["fill_ratio"].clip(0, 1)          # a few stations report more bikes than docks
    df["hour_label"] = df["hour"].map(lambda h: HOUR_LABELS[h])
    df = df.sort_values("hour")

    fig = px.scatter_map(
        df, lat="lat", lon="lon",
        color="fill_ratio", range_color=[0, 1],
        color_continuous_scale="RdYlBu",                    # red = empty, blue = full
        animation_frame="hour_label",
        category_orders={"hour_label": HOUR_LABELS},
        hover_name="name",
        hover_data={"lat": False, "lon": False, "hour_label": False,
                    "fill_ratio": ":.0%", "avg_bikes": ":.1f", "capacity": True},
        labels={"fill_ratio": "Fill", "avg_bikes": "Avg bikes", "capacity": "Docks"},
        center=NYC_CENTER, zoom=11, map_style=MAP_STYLE, height=800,
    )
    fig.update_traces(marker=dict(size=7, opacity=0.9))
    fig.update_layout(
        margin=dict(l=10, r=10, t=130, b=40),
        coloraxis_colorbar=dict(title="Bikes ÷ docks", tickformat=".0%", thickness=14, len=0.6),
    )

    # Smoother playback + clearer slider label
    if fig.layout.updatemenus:
        play = fig.layout.updatemenus[0].buttons[0].args[1]
        play["frame"]["duration"] = 700
        play["transition"]["duration"] = 300
    if fig.layout.sliders:
        fig.layout.sliders[0].currentvalue.prefix = "Hour: "
        fig.layout.sliders[0].pad.t = 10

    add_titles(fig, "A day in the life of NYC's bike supply",
               "Red stations are out of bikes, blue stations are out of docks · press ▶ to play")
    add_source(fig, source_note, y=-0.01)
    return fig


hourly_df = spark.sql(f"""
    SELECT i.name, i.lat, i.lon,
           hour(h.hour_ny)                              AS hour,
           avg(h.avg_bikes / nullif(i.capacity, 0))     AS fill_ratio,
           avg(h.avg_bikes)                             AS avg_bikes,
           any_value(i.capacity)                        AS capacity
    FROM {GOLD}.station_hour h
    JOIN {SILVER}.station_info   i USING (station_id)
    JOIN {GOLD}.station_profile  p USING (station_id)
    WHERE p.station_type <> 'Out of service'
    GROUP BY i.name, i.lat, i.lon, hour(h.hour_ny)
    ORDER BY hour
""").toPandas()

fig_anim = build_hourly_animation(hourly_df, SOURCE_NOTE)
fig_anim.show()
export(fig_anim, "03_hourly_animation_map")

## 4. Worst-stations heatmap

The 40 operating stations that spend the most time empty, hour by hour. Dark only at certain hours → needs a timed rebalancing run. Dark all day → structural shortage.

In [0]:
def build_swing_heatmap(df, source_note, top_n=40):
    df = df.copy()
    df["fill_ratio"] = df["fill_ratio"].clip(0, 1)

    grid = (df.pivot_table(index="name", columns="hour", values="fill_ratio")
              .reindex(columns=range(24)))                      # all 24 hours, even if some are missing

    # Pick the stations with the biggest daily swing, then sort by when they bottom out
    swing  = grid.max(axis=1) - grid.min(axis=1)
    grid   = grid.loc[swing.nlargest(top_n).index]
    trough = grid.idxmin(axis=1)
    order  = sorted(grid.index, key=lambda s: (trough[s], -swing[s]))
    grid   = grid.loc[order]

    # Label rows with the low-point hour so the pattern is readable without hovering
    labels = [f"{name}  ·  low {HOUR_LABELS[trough[name]]}" for name in grid.index]
    hover_swing = [[swing[name]] * 24 for name in grid.index]

    fig = go.Figure(go.Heatmap(
        z=grid.values, x=HOUR_LABELS, y=labels,
        customdata=hover_swing,
        colorscale="RdYlBu", zmin=0, zmax=1,
        xgap=1, ygap=1, hoverongaps=False,
        colorbar=dict(title="Bikes ÷ docks", tickformat=".0%", thickness=14, len=0.6,
                      tickvals=[0, 0.5, 1], ticktext=["0% (empty)", "50%", "100% (full)"]),
        hovertemplate=("<b>%{y}</b><br>%{x}: %{z:.0%} full"
                       "<br>Daily swing: %{customdata:.0%}<extra></extra>"),
    ))

    for x0, x1 in [(7, 9), (16, 18)]:                          # outline the commute peaks
        fig.add_shape(type="rect", xref="x", yref="paper",
                      x0=x0 - 0.5, x1=x1 + 0.5, y0=0, y1=1,
                      line=dict(color=INK, width=1.5, dash="dot"))

    fig.update_yaxes(autorange="reversed", showgrid=False, tickfont=dict(size=11), linecolor="white")
    fig.update_xaxes(tickvals=HOUR_LABELS[::3], side="top", linecolor="white", title_text="")
    fig.update_layout(height=max(650, 22 * len(grid) + 260),
                      margin=dict(l=320, r=30, t=170, b=70))

    add_titles(fig, "The stations that swing hardest every day",
               "Fill ratio by hour for the 40 most volatile stations · sorted by when they run lowest · "
               "dotted boxes mark the commute rushes")
    add_source(fig, source_note, y=-0.03)
    return fig


heat_df = spark.sql(f"""
    SELECT i.name,
           hour(h.hour_ny)                           AS hour,
           avg(h.avg_bikes / nullif(i.capacity, 0))  AS fill_ratio
    FROM {GOLD}.station_hour h
    JOIN {SILVER}.station_info  i USING (station_id)
    JOIN {GOLD}.station_profile p USING (station_id)
    WHERE p.station_type <> 'Out of service'
      AND i.capacity > 0
    GROUP BY i.name, hour(h.hour_ny)
""").toPandas()

fig_heat = build_swing_heatmap(heat_df, SOURCE_NOTE)
fig_heat.show()
export(fig_heat, "04_daily_swing_heatmap")

## 5. Exported files

Download from **Catalog → citibike → gold → exports**, then copy into your repo's `docs/` folder for GitHub Pages or your website.

In [0]:
for f in sorted(dbutils.fs.ls(EXPORT_DIR), key=lambda f: f.name):
    print(f"{f.name:<35} {f.size / 1e6:6.2f} MB")